# Multimodal SLM via Embedding Distillation

**Architecture:** `[Frozen CLIP ViT-B/16] → [Trainable MLP Adapter] → [Frozen Phi-2 4-bit]`

**Drive layout:**
```
My Drive/Projects/Multimodal SLM/
    data/coco/          ← downloaded once, reused every session
    checkpoints/        ← per-experiment subdirs
```

**Run order:**
- Run every cell in Phase 1 at the start of every new session
- Phase 1.6b (train2017 download) only needed for `baseline` config
- Use `smoke_test` first, then `baseline` for the full run

---
## Phase 1 · Session Setup

In [ ]:
# ── 1.1  Mount Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/Projects/Multimodal SLM"
COCO_ROOT  = f"{DRIVE_ROOT}/data/coco"
CKPT_ROOT  = f"{DRIVE_ROOT}/checkpoints"

os.makedirs(COCO_ROOT,  exist_ok=True)
os.makedirs(CKPT_ROOT,  exist_ok=True)
print(f"Drive root : {DRIVE_ROOT}")
print(f"COCO root  : {COCO_ROOT}")
print(f"Ckpt root  : {CKPT_ROOT}")

In [ ]:
# ── 1.2  Clone repo (or pull latest if already cloned) ───────────────────────
# All .py source files live in GitHub. The notebook only contains glue code.
# After editing any .py file and pushing, re-run this cell to get updates
# without restarting the runtime.

import sys, importlib

REPO_URL = "https://github.com/UtkarshC99/multimodal-slm"
REPO_DIR = "/content/multimodal-slm"

if os.path.isdir(f"{REPO_DIR}/.git"):
    print("Repo already cloned — pulling latest...")
    !git -C {REPO_DIR} pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
%cd {REPO_DIR}

# Verify key modules are importable
import utils.config, models.adapter, data.dataset, training.trainer, evaluation.evaluator
print("All modules importable ✓")

In [ ]:
# ── 1.3  Install dependencies ────────────────────────────────────────────────
!pip install -q -r requirements.txt
print("Dependencies installed ✓")

In [ ]:
# ── 1.4  Verify GPU and set device ───────────────────────────────────────────
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be extremely slow.")

In [ ]:
# ── 1.5  Set seeds ───────────────────────────────────────────────────────────
import random, numpy as np, os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("Seeds fixed ✓")

In [ ]:
# ── 1.6  Download COCO val2017 (once — persists on Drive) ────────────────────
# Checks Drive first; only downloads if not already present.
# val2017 is needed for every config (smoke_test trains AND validates on it).

ann_dir = os.path.join(COCO_ROOT, "annotations")
val_dir = os.path.join(COCO_ROOT, "val2017")

if not os.path.isdir(ann_dir):
    print("Downloading COCO annotations (~240 MB)...")
    !wget -q -nc http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P {COCO_ROOT}
    !unzip -q -n {COCO_ROOT}/annotations_trainval2017.zip -d {COCO_ROOT}
    !rm -f {COCO_ROOT}/annotations_trainval2017.zip
    print("Annotations downloaded ✓")
else:
    print(f"Annotations already on Drive ✓")

n_val = len(os.listdir(val_dir)) if os.path.isdir(val_dir) else 0
if n_val < 5000:
    print("Downloading COCO val2017 images (~1 GB, ~3 min)...")
    !wget -q -nc http://images.cocodataset.org/zips/val2017.zip -P {COCO_ROOT}
    !unzip -q -n {COCO_ROOT}/val2017.zip -d {COCO_ROOT}
    !rm -f {COCO_ROOT}/val2017.zip
    n_val = len(os.listdir(val_dir))
    print(f"val2017 downloaded: {n_val:,} images ✓")
else:
    print(f"val2017 already on Drive: {n_val:,} images ✓")

In [ ]:
# ── 1.6b  Download COCO train2017 — BASELINE ONLY, skip for smoke_test ───────
# train2017 is ~18 GB and takes ~20 min. Only run this cell when you are
# ready to run cfg = get_config("baseline"). It persists on Drive.

train_dir = os.path.join(COCO_ROOT, "train2017")
n_train   = len(os.listdir(train_dir)) if os.path.isdir(train_dir) else 0

if n_train >= 100000:
    print(f"train2017 already on Drive: {n_train:,} images ✓")
else:
    print(f"Downloading COCO train2017 (~18 GB, ~20 min on Colab)...")
    !wget -q -nc http://images.cocodataset.org/zips/train2017.zip -P {COCO_ROOT}
    !unzip -q -n {COCO_ROOT}/train2017.zip -d {COCO_ROOT}
    !rm -f {COCO_ROOT}/train2017.zip
    n_train = len(os.listdir(train_dir))
    print(f"train2017 downloaded: {n_train:,} images ✓")

In [ ]:
# ── 1.7  Load experiment config ──────────────────────────────────────────────
# Switch between configs here — everything else adapts automatically.
#
#   "smoke_test"  : 5k samples, val2017, 5 epochs, MLP+contrastive  (~10 min)
#   "baseline"    : full train2017, 3 epochs, MLP+contrastive        (~10-12 h)
#
# Ablation configs (run after baseline to isolate each variable):
#   "ablation_linear"              : no MLP nonlinearity
#   "ablation_no_contrastive"      : LM loss only — shows mode collapse
#   "ablation_perceiver"           : patch tokens + perceiver adapter
#   "ablation_contrastive_weight"  : contrastive weight sweep

from utils.config import get_config, COCO_ROOT as _CFG_COCO

cfg = get_config("smoke_test")   # ← change this line only

# Paths are set in config.py but override here in case Drive layout differs
cfg.coco_root  = COCO_ROOT
cfg.output_dir = os.path.join(CKPT_ROOT, cfg.experiment_name)
os.makedirs(cfg.output_dir, exist_ok=True)
print(f"\nCheckpoints → {cfg.output_dir}")

---
## Phase 2 · Model Setup & Sanity Check

In [ ]:
# ── 2.1  Build VisionLanguageModel ───────────────────────────────────────────
# Reloading the model in a session that already has one loaded? Run the del
# block first to free VRAM, then re-run this cell.

import gc
if "model" in dir() and model is not None:
    del model
    torch.cuda.empty_cache()
    gc.collect()
    print("Previous model cleared from VRAM")

from models.adapter import VisionLanguageModel

adapter_kwargs = {}
if cfg.adapter_type == "mlp":
    adapter_kwargs["hidden_scale"] = cfg.mlp_hidden_scale
elif cfg.adapter_type == "perceiver":
    adapter_kwargs["num_queries"] = cfg.perceiver_num_queries
    adapter_kwargs["num_heads"]   = cfg.perceiver_num_heads
    adapter_kwargs["num_layers"]  = cfg.perceiver_num_layers

model = VisionLanguageModel(
    vision_model_name = cfg.vision_model_name,
    lm_model_name     = cfg.lm_model_name,
    adapter_type      = cfg.adapter_type,
    use_cls_only      = cfg.use_cls_only,
    vision_dim        = cfg.vision_dim,
    text_dim          = cfg.text_dim,
    use_4bit          = cfg.use_4bit,
    **adapter_kwargs,
)

# Move the two non-quantized components to GPU
# (LLM stays where device_map="auto" placed it — do not call .to() on it)
model.vision_encoder = model.vision_encoder.to(device)
model.adapter        = model.adapter.to(device)

total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
used_vram  = torch.cuda.memory_allocated() / 1e9
free_vram  = total_vram - used_vram

print(f"\nAdapter params    : {model.num_trainable_params():,}")
print(f"VRAM used         : {used_vram:.2f} GB / {total_vram:.2f} GB")
print(f"VRAM free         : {free_vram:.2f} GB", end="  ")
print("✓ enough for training" if free_vram > 3.5 else "⚠ tight — reduce batch_size if OOM")

In [ ]:
# ── 2.2  Load tokenizer and image processor ───────────────────────────────────
from transformers import AutoTokenizer
from data.dataset import get_processor

tokenizer = AutoTokenizer.from_pretrained(cfg.lm_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

processor = get_processor(cfg.vision_model_name)
print(f"Tokenizer : {cfg.lm_model_name}  vocab={tokenizer.vocab_size:,}")
print(f"Processor : {type(processor).__name__}")

In [ ]:
# ── 2.3  Forward pass sanity check ────────────────────────────────────────────
import requests
from io import BytesIO
from PIL import Image

url       = "http://images.cocodataset.org/val2017/000000039769.jpg"
pil_image = Image.open(BytesIO(requests.get(url, timeout=10).content)).convert("RGB")
display(pil_image)

pixel_values  = processor(images=pil_image, return_tensors="pt")["pixel_values"].to(device)
inputs        = tokenizer("What is in this image?", return_tensors="pt")
input_ids     = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

with torch.no_grad():
    vis_emb = model.encode_images(pixel_values)

print(f"pixel_values  : {tuple(pixel_values.shape)}  range [{pixel_values.min():.2f}, {pixel_values.max():.2f}]")
print(f"input_ids     : {tuple(input_ids.shape)}  → {tokenizer.convert_ids_to_tokens(input_ids[0].tolist())}")
print(f"vis_emb       : {tuple(vis_emb.shape)}  → (batch, visual_tokens, lm_dim={cfg.text_dim})")
print("\nShapes ✓")

In [ ]:
# ── 2.4  Untrained generation (expect incoherent output) ─────────────────────
with torch.no_grad():
    gen_ids = model.generate(
        pixel_values   = pixel_values,
        input_ids      = input_ids,
        attention_mask = attention_mask,
        max_new_tokens = 40,
        do_sample      = False,
    )

new_tokens  = gen_ids[0][input_ids.shape[-1]:]
output_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
print("[Untrained output — incoherent is expected]")
print(output_text)

---
## Phase 3 · Alignment Training

In [ ]:
# ── 3.1  Build datasets ──────────────────────────────────────────────────────
# train_split is set in config:
#   smoke_test → "val"   (uses val2017, no train download needed)
#   baseline   → "train" (requires train2017 on Drive from cell 1.6b)

from data.dataset import COCOCaptionsDataset, build_dataloader

train_dataset = COCOCaptionsDataset(
    coco_root   = cfg.coco_root,
    split       = cfg.train_split,
    tokenizer   = tokenizer,
    processor   = processor,
    max_length  = cfg.max_caption_length,
    max_samples = cfg.max_train_samples,
)

val_dataset = COCOCaptionsDataset(
    coco_root   = cfg.coco_root,
    split       = "val",
    tokenizer   = tokenizer,
    processor   = processor,
    max_length  = cfg.max_caption_length,
    max_samples = cfg.max_val_samples,
)

train_loader = build_dataloader(train_dataset, batch_size=cfg.batch_size,
                                shuffle=True,  num_workers=cfg.num_workers)
val_loader   = build_dataloader(val_dataset,   batch_size=cfg.batch_size,
                                shuffle=False, num_workers=cfg.num_workers)

total_steps = (len(train_loader) // cfg.grad_accum_steps) * cfg.num_epochs
print(f"Train : {len(train_dataset):,} samples  ({len(train_loader)} batches)")
print(f"Val   : {len(val_dataset):,} samples  ({len(val_loader)} batches)")
print(f"Total optimiser steps over {cfg.num_epochs} epochs: {total_steps}")

In [ ]:
# ── 3.2  Inspect a batch ─────────────────────────────────────────────────────
batch = next(iter(train_loader))
print("Keys           :", list(batch.keys()))
print("pixel_values   :", batch["pixel_values"].shape)
print("input_ids      :", batch["input_ids"].shape)
print("labels         :", batch["labels"].shape)
print("Sample caption :", batch["caption"][0])

In [ ]:
# ── 3.3  Pre-training CKA baseline ───────────────────────────────────────────
# Establishes where the embedding alignment starts. Target after training: > 0.5.
# The improvement delta is what matters most, not the absolute value.

from evaluation.evaluator import EmbeddingAnalyzer

analyzer   = EmbeddingAnalyzer()
cka_before = analyzer.compute_cka(model, val_loader, device=device, max_batches=20)
print(f"\nPre-training CKA : {cka_before:.4f}")

In [ ]:
# ── 3.4  t-SNE before training ───────────────────────────────────────────────
# Save to Drive so you can compare across sessions.

tsne_dir = os.path.join(DRIVE_ROOT, "plots", cfg.experiment_name)
os.makedirs(tsne_dir, exist_ok=True)

fig_before = analyzer.tsne_plot(
    model, val_loader, device=device, max_batches=10,
    save_path=os.path.join(tsne_dir, "tsne_before.png"),
)

In [ ]:
# ── 3.5  Initialise trainer ───────────────────────────────────────────────────
from training.trainer import Trainer

trainer = Trainer(
    model              = model,
    train_loader       = train_loader,
    val_loader         = val_loader,
    tokenizer          = tokenizer,       # needed for per-epoch BLEU-1
    output_dir         = cfg.output_dir,
    num_epochs         = cfg.num_epochs,
    learning_rate      = cfg.learning_rate,
    weight_decay       = cfg.weight_decay,
    warmup_ratio       = cfg.warmup_ratio,
    grad_accum_steps   = cfg.grad_accum_steps,
    max_grad_norm      = cfg.max_grad_norm,
    use_contrastive    = cfg.use_contrastive,
    contrastive_weight = cfg.contrastive_weight,
    use_l2             = cfg.use_l2,
    l2_weight          = cfg.l2_weight,
    log_every          = cfg.log_every,
    save_every         = cfg.save_every,
    bleu_batches       = 50,              # val batches sampled per epoch for BLEU-1
    use_wandb          = cfg.use_wandb,
)

# To resume from a mid-run checkpoint:
# trainer.load_checkpoint(f"{cfg.output_dir}/adapter_step500.pt")

In [ ]:
# ── 3.6  TRAIN ───────────────────────────────────────────────────────────────
# Expected times on T4:
#   smoke_test  (5k samples × 5 epochs)  : ~10 min
#   baseline    (118k samples × 3 epochs): ~10-12 h
#
# Per-epoch output: val_loss + BLEU-1 (quick proxy — see Phase 4 for full eval)
# Checkpoints saved to Drive at end of each epoch and every save_every steps.

import gc
torch.cuda.empty_cache(); gc.collect()
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"Free VRAM before training: {free:.2f} GB")

trainer.train()

In [ ]:
# ── 3.7  Load best checkpoint ─────────────────────────────────────────────────
# Loads the final saved adapter weights back into the model for evaluation.
# If you want a specific epoch, change "final" to e.g. "epoch2".

ckpt_path = os.path.join(cfg.output_dir, "adapter_final.pt")
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.adapter.load_state_dict(ckpt["adapter_state_dict"])
    print(f"Loaded: {ckpt_path}  (step {ckpt['global_step']})")
else:
    print(f"Not found: {ckpt_path}")
    print("Available:", sorted(os.listdir(cfg.output_dir)))

---
## Phase 4 · Evaluation

In [ ]:
# ── 4.1  Qualitative captions ─────────────────────────────────────────────────
# Images pulled from the local COCO val2017 directory using annotation file
# so labels are guaranteed to match the actual image content.

import json, random
import matplotlib.pyplot as plt
from PIL import Image as PILImage

ann_path = os.path.join(COCO_ROOT, "annotations", "captions_val2017.json")
with open(ann_path) as f:
    coco_ann = json.load(f)

# Build image_id → (filename, one caption) map
id2info = {img["id"]: img["file_name"] for img in coco_ann["images"]}
id2cap  = {}
for ann in coco_ann["annotations"]:
    if ann["image_id"] not in id2cap:
        id2cap[ann["image_id"]] = ann["caption"]

# Sample 4 random images
sample_ids = random.sample(list(id2cap.keys()), 4)

model.eval()
prompt_str = "Describe this image:"
prompt_ids = tokenizer(prompt_str, return_tensors="pt")["input_ids"].to(device)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, img_id in zip(axes, sample_ids):
    img_path = os.path.join(COCO_ROOT, "val2017", id2info[img_id])
    img      = PILImage.open(img_path).convert("RGB")
    pv       = processor(images=img, return_tensors="pt")["pixel_values"].to(device)

    with torch.no_grad():
        gen = model.generate(
            pixel_values   = pv,
            input_ids      = prompt_ids,
            attention_mask = torch.ones_like(prompt_ids),
            max_new_tokens = 50,
            do_sample      = False,
        )
    caption = tokenizer.decode(gen[0][prompt_ids.shape[-1]:], skip_special_tokens=True)

    ax.imshow(img)
    ax.axis("off")
    gt_short = id2cap[img_id][:60] + "..." if len(id2cap[img_id]) > 60 else id2cap[img_id]
    ax.set_title(f"GT: {gt_short}\n\nPred: {caption[:70]}", fontsize=7)

plt.tight_layout()
plot_path = os.path.join(DRIVE_ROOT, "plots", cfg.experiment_name, "qualitative.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved → {plot_path}")

In [ ]:
# ── 4.2  CKA after training ───────────────────────────────────────────────────
cka_after = analyzer.compute_cka(model, val_loader, device=device, max_batches=20)
print(f"CKA before : {cka_before:.4f}")
print(f"CKA after  : {cka_after:.4f}")
print(f"Delta      : {cka_after - cka_before:+.4f}")
print()
if cka_after - cka_before < 0.02:
    print("⚠ Small improvement — check training logs for loss decrease")
elif cka_after > 0.5:
    print("✓ Good alignment — ready for baseline or ablation runs")
else:
    print("↑ Improving — consider more epochs or data")

In [ ]:
# ── 4.3  t-SNE after training ────────────────────────────────────────────────
fig_after = analyzer.tsne_plot(
    model, val_loader, device=device, max_batches=10,
    save_path=os.path.join(tsne_dir, "tsne_after.png"),
)

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(plt.imread(os.path.join(tsne_dir, "tsne_before.png")))
axes[0].set_title("Before Training"); axes[0].axis("off")
axes[1].imshow(plt.imread(os.path.join(tsne_dir, "tsne_after.png")))
axes[1].set_title("After Training");  axes[1].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(tsne_dir, "tsne_comparison.png"), dpi=150)
plt.show()

In [ ]:
# ── 4.4  Cosine similarity stats ─────────────────────────────────────────────
# Diagnostic: matched pairs should have higher similarity than unmatched.
# A good adapter: alignment_gap > 0.1

sim_stats = analyzer.cosine_similarity_stats(model, val_loader, device=device, max_batches=50)

In [ ]:
# ── 4.5  BLEU-1 (full val set) ───────────────────────────────────────────────
# Full captioning evaluation — compare against the per-epoch quick BLEU-1
# logged during training to verify they track together.

from evaluation.evaluator import CaptionEvaluator

cap_eval    = CaptionEvaluator(model, tokenizer, processor, device=device)
cap_records = cap_eval.generate_captions(val_loader, max_new_tokens=40)
bleu1       = cap_eval.bleu1_approx(cap_records)
print(f"BLEU-1 (full val): {bleu1:.4f}")

print("\nSample predictions:")
for rec in cap_records[:5]:
    print(f"  GT  : {rec['ground_truth']}")
    print(f"  Pred: {rec['prediction']}")
    print()

In [ ]:
# ── 4.6  VQA Evaluation (optional) ──────────────────────────────────────────
# Download instructions (run once, save to Drive):
#   wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip
#   wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip
#   wget http://images.cocodataset.org/zips/val2014.zip
# Point VQA_ROOT at the Drive folder where you extract them.

VQA_ROOT = os.path.join(DRIVE_ROOT, "data", "vqa")

if os.path.isdir(VQA_ROOT):
    from data.dataset import VQAv2Dataset
    from evaluation.evaluator import VQAEvaluator

    vqa_dataset = VQAv2Dataset(
        vqa_root    = VQA_ROOT,
        processor   = processor,
        tokenizer   = tokenizer,
        max_samples = 2000,
    )
    vqa_loader  = build_dataloader(vqa_dataset, batch_size=8, shuffle=False)
    vqa_eval    = VQAEvaluator(model, tokenizer, processor, device=device)
    vqa_results = vqa_eval.evaluate(vqa_loader, max_new_tokens=5)
else:
    print(f"VQA data not found at {VQA_ROOT}")
    print("Download instructions in the cell comment above.")
    vqa_results = {}

---
## Phase 5 · Ablation Studies

Run each ablation config independently. All use the same 5k val2017 split so results are directly comparable to smoke_test. The only variable that changes between runs is the one being tested.

In [ ]:
# ── 5.1  Run a single ablation ───────────────────────────────────────────────
# Change ABLATION_NAME to run each config. Recommended order:
#   1. "ablation_no_contrastive"      — shows what training looks like without it
#   2. "ablation_linear"              — compares MLP vs linear adapter
#   3. "ablation_perceiver"           — patch tokens + cross-attention
#   4. "ablation_contrastive_weight"  — weight sensitivity
#
# Results are saved to Drive checkpoints and plots dirs per experiment name.

ABLATION_NAME = "ablation_no_contrastive"   # ← change this

import gc, importlib
import utils.config, models.adapter, training.trainer

# Reload modules in case you pushed code changes
for mod in [utils.config, models.adapter, training.trainer]:
    importlib.reload(mod)
from utils.config import get_config
from models.adapter import VisionLanguageModel
from training.trainer import Trainer
from data.dataset import COCOCaptionsDataset, build_dataloader

ab_cfg             = get_config(ABLATION_NAME)
ab_cfg.coco_root   = COCO_ROOT
ab_cfg.output_dir  = os.path.join(CKPT_ROOT, ABLATION_NAME)
os.makedirs(ab_cfg.output_dir, exist_ok=True)

# ── Build model ────────────────────────────────────────────────────────────
kw = {}
if ab_cfg.adapter_type == "mlp":
    kw["hidden_scale"] = ab_cfg.mlp_hidden_scale
elif ab_cfg.adapter_type == "perceiver":
    kw["num_queries"] = ab_cfg.perceiver_num_queries
    kw["num_heads"]   = ab_cfg.perceiver_num_heads
    kw["num_layers"]  = ab_cfg.perceiver_num_layers

ab_model = VisionLanguageModel(
    vision_model_name = ab_cfg.vision_model_name,
    lm_model_name     = ab_cfg.lm_model_name,
    adapter_type      = ab_cfg.adapter_type,
    use_cls_only      = ab_cfg.use_cls_only,
    vision_dim        = ab_cfg.vision_dim,
    text_dim          = ab_cfg.text_dim,
    use_4bit          = True,
    **kw,
)
ab_model.vision_encoder = ab_model.vision_encoder.to(device)
ab_model.adapter        = ab_model.adapter.to(device)

# ── Datasets ───────────────────────────────────────────────────────────────
ab_train_ds = COCOCaptionsDataset(
    ab_cfg.coco_root, ab_cfg.train_split, tokenizer, processor,
    max_samples=ab_cfg.max_train_samples
)
ab_val_ds = COCOCaptionsDataset(
    ab_cfg.coco_root, "val", tokenizer, processor,
    max_samples=ab_cfg.max_val_samples
)
ab_train_dl = build_dataloader(ab_train_ds, batch_size=2, num_workers=0)
ab_val_dl   = build_dataloader(ab_val_ds,   batch_size=2, num_workers=0, shuffle=False)

# ── Train ──────────────────────────────────────────────────────────────────
ab_trainer = Trainer(
    model              = ab_model,
    train_loader       = ab_train_dl,
    val_loader         = ab_val_dl,
    tokenizer          = tokenizer,
    output_dir         = ab_cfg.output_dir,
    num_epochs         = ab_cfg.num_epochs,
    learning_rate      = ab_cfg.learning_rate,
    warmup_ratio       = ab_cfg.warmup_ratio,
    grad_accum_steps   = ab_cfg.grad_accum_steps,
    use_contrastive    = ab_cfg.use_contrastive,
    contrastive_weight = ab_cfg.contrastive_weight,
    log_every          = ab_cfg.log_every,
    save_every         = ab_cfg.save_every,
    bleu_batches       = 50,
)
ab_cka_before = analyzer.compute_cka(ab_model, ab_val_dl, device=device, max_batches=10)
ab_trainer.train()
ab_cka_after  = analyzer.compute_cka(ab_model, ab_val_dl, device=device, max_batches=10)

print(f"\n{ABLATION_NAME}  CKA {ab_cka_before:.4f} → {ab_cka_after:.4f}  "
      f"(delta {ab_cka_after - ab_cka_before:+.4f})")

del ab_model
torch.cuda.empty_cache(); gc.collect()

In [ ]:
# ── 5.2  Compare all completed ablations ─────────────────────────────────────
# Reads final checkpoints from Drive and reports CKA for each.
# Only configs that have a completed "adapter_final.pt" are included.

import json

ABLATION_NAMES = [
    "smoke_test",
    "ablation_linear",
    "ablation_no_contrastive",
    "ablation_perceiver",
    "ablation_contrastive_weight",
]

results = {}
for name in ABLATION_NAMES:
    ckpt = os.path.join(CKPT_ROOT, name, "adapter_final.pt")
    if not os.path.exists(ckpt):
        print(f"  {name}: no checkpoint yet — skip")
        continue

    ab_cfg = get_config(name)
    kw = {}
    if ab_cfg.adapter_type == "mlp":
        kw["hidden_scale"] = ab_cfg.mlp_hidden_scale
    elif ab_cfg.adapter_type == "perceiver":
        kw["num_queries"] = ab_cfg.perceiver_num_queries
        kw["num_heads"]   = ab_cfg.perceiver_num_heads
        kw["num_layers"]  = ab_cfg.perceiver_num_layers

    ab_model = VisionLanguageModel(
        vision_model_name=ab_cfg.vision_model_name, lm_model_name=ab_cfg.lm_model_name,
        adapter_type=ab_cfg.adapter_type, use_cls_only=ab_cfg.use_cls_only,
        vision_dim=ab_cfg.vision_dim, text_dim=ab_cfg.text_dim, use_4bit=True, **kw,
    )
    ab_model.vision_encoder = ab_model.vision_encoder.to(device)
    ab_model.adapter        = ab_model.adapter.to(device)
    state = torch.load(ckpt, map_location="cpu")
    ab_model.adapter.load_state_dict(state["adapter_state_dict"])

    ab_val_ds = COCOCaptionsDataset(COCO_ROOT, "val", tokenizer, processor, max_samples=500)
    ab_val_dl = build_dataloader(ab_val_ds, batch_size=2, num_workers=0, shuffle=False)
    cka = analyzer.compute_cka(ab_model, ab_val_dl, device=device, max_batches=10)
    results[name] = {"adapter": ab_cfg.adapter_type, "params": ab_model.num_trainable_params(),
                     "contrastive": ab_cfg.use_contrastive, "cka": cka}
    del ab_model; torch.cuda.empty_cache(); gc.collect()

print(f"\n{'Config':<35} {'Adapter':<12} {'Params':>10}  {'CKA':>6}")
print("-" * 70)
for name, r in results.items():
    print(f"{name:<35} {r['adapter']:<12} {r['params']:>10,}  {r['cka']:>6.4f}")

In [ ]:
# ── 5.3  Plot ablation comparison ────────────────────────────────────────────
import matplotlib.pyplot as plt

if results:
    names = list(results.keys())
    ckas  = [results[n]["cka"] for n in names]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(names, ckas, color=["steelblue" if n == "smoke_test" else "tomato" for n in names])
    ax.axhline(0.5, color="green", linestyle="--", linewidth=1, label="Target CKA = 0.5")
    ax.set_ylabel("CKA Similarity")
    ax.set_title("Embedding Alignment by Config")
    ax.legend()
    plt.xticks(rotation=20, ha="right")
    for bar, v in zip(bars, ckas):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plot_path = os.path.join(DRIVE_ROOT, "plots", "ablation_comparison.png")
    os.makedirs(os.path.dirname(plot_path), exist_ok=True)
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"Saved → {plot_path}")

In [ ]:
# ── 5.4  Failure mode analysis ───────────────────────────────────────────────
# Only useful after VQA evaluation (cell 4.6) has been run.
# Breaks accuracy down by question type to identify where the model fails.

if "vqa_results" in dir() and vqa_results.get("results"):
    from collections import defaultdict
    type_stats = defaultdict(lambda: {"correct": 0, "total": 0})
    for r in vqa_results["results"]:
        q = r.get("question", "").lower()
        if q.startswith("how many"):                    qtype = "counting"
        elif "color" in q:                              qtype = "color/attribute"
        elif q.startswith(("is there","are there")):   qtype = "presence"
        elif q.startswith("where"):                    qtype = "spatial"
        else:                                          qtype = "other"
        type_stats[qtype]["total"]   += 1
        type_stats[qtype]["correct"] += r["correct"]

    print(f"{'Question type':<20} {'Accuracy':>8}  {'Count':>6}")
    print("-" * 40)
    for qtype, s in sorted(type_stats.items()):
        acc = s["correct"] / max(1, s["total"])
        print(f"{qtype:<20} {acc:>8.3f}  {s['total']:>6,}")
else:
    print("Run cell 4.6 (VQA Evaluation) first.")

---
## Appendix · Utilities

In [ ]:
# ── A1  Pull latest code and reload modules ───────────────────────────────────
# Use this mid-session after pushing a fix to GitHub.
# Reloads all modules without restarting the runtime or reloading the model.

!git -C {REPO_DIR} pull

import importlib
import utils.config, models.adapter, data.dataset, training.trainer, evaluation.evaluator
for mod in [utils.config, models.adapter, data.dataset, training.trainer, evaluation.evaluator]:
    importlib.reload(mod)
print("Modules reloaded ✓")

In [ ]:
# ── A2  VRAM status ──────────────────────────────────────────────────────────
alloc    = torch.cuda.memory_allocated()  / 1e9
reserved = torch.cuda.memory_reserved()   / 1e9
total    = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Allocated : {alloc:.2f} GB")
print(f"Reserved  : {reserved:.2f} GB")
print(f"Total     : {total:.2f} GB")
print(f"Free      : {total - alloc:.2f} GB")

In [ ]:
# ── A3  Quick inference on your own image ─────────────────────────────────────
from google.colab import files
from io import BytesIO

uploaded = files.upload()
for fname, content in uploaded.items():
    img = PILImage.open(BytesIO(content)).convert("RGB")
    pv  = processor(images=img, return_tensors="pt")["pixel_values"].to(device)
    p   = tokenizer("Describe this image:", return_tensors="pt")["input_ids"].to(device)
    model.eval()
    with torch.no_grad():
        gen = model.generate(pv, p, torch.ones_like(p), max_new_tokens=60)
    print(tokenizer.decode(gen[0][p.shape[-1]:], skip_special_tokens=True))

In [ ]:
# ── A4  Export adapter weights ───────────────────────────────────────────────
export_path = os.path.join(cfg.output_dir, "adapter_export.pt")
torch.save(model.adapter.state_dict(), export_path)
size_mb = os.path.getsize(export_path) / 1e6
print(f"Exported → {export_path}  ({size_mb:.1f} MB)")